# Datartchive
## A Data-Driven Portrait of Artists in the PHAROS Archives

**Course:** Information Visualization course (a.a. 2025/2026) within the Digital Humanities and Digital Knowledge Master's Degree at Alma Mater Studiorum - University of Bologna.

**Data sources:**
- [artresearch.net](https://artresearch.net) — PHAROS Linked Open Data (primary LOD source)
- [Getty ULAN](https://vocab.getty.edu) — nationality, gender, and biographical enrichment (secondary LOD source)

---

### Abstract

This notebook analyses the persons documented across five PHAROS member institutions —
Bildarchiv Foto Marburg (MIDAS), Frick Art Research Library, Fondazione Zeri, Paul Mellon Centre,
and the Warburg Institute — through four research questions:
(1) How are persons distributed across PHAROS member institutions?
(2) What is the nationality distribution of persons in the archives?
(3) How is gender represented, and does this vary over time?
(4) Where were persons born and where did they die — what does this reveal about geographic mobility?

Data is queried live from the artresearch.net SPARQL endpoint using CIDOC-CRM predicates
and enriched with biographical records from the Getty ULAN linked open dataset.

> **Methodological note:** 
>At the time of data collection, person records were retrievable from five institutions via the artresearch.net SPARQL endpoint. 
>Other PHAROS members (Courtauld, RKD, NGA, Yale, Hertziana, KHI, Dumbarton Oaks, I Tatti) returned no results — this may reflect differences in data modelling, endpoint availability, or ongoing data integration.

---

## 0. Setup

In [ ]:
# !pip install SPARQLWrapper pandas plotly

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from SPARQLWrapper import SPARQLWrapper, JSON
import os, time

In [ ]:


PHAROS_ENDPOINT   = "https://artresearch.net/sparql"
GETTY_ENDPOINT    = "https://vocab.getty.edu/sparql"

def run_query(endpoint_url, query, agent="PHAROS-InfoVis-Project/1.0"):
    """
    Execute a SPARQL SELECT query and return results as a pandas DataFrame.

    Parameters
    ----------
    endpoint_url : str — SPARQL endpoint URL
    query        : str — SPARQL SELECT query string
    agent        : str — User-Agent header (required by some endpoints)

    Returns
    -------
    pd.DataFrame with one column per SELECT variable, empty DataFrame if no results
    """
    sparql = SPARQLWrapper(endpoint_url)
    sparql.addCustomHttpHeader("User-Agent", agent)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results["results"]["bindings"]
    if not bindings:
        return pd.DataFrame()
    cols = results["head"]["vars"]
    rows = [{c: b[c]["value"] if c in b else None for c in cols} for b in bindings]
    return pd.DataFrame(rows, columns=cols)

def fetch_all_paginated(query, page_size=5000):
    """
    Fetch all SPARQL results using LIMIT/OFFSET pagination to avoid endpoint timeouts.

    Parameters
    ----------
    query     : str — SPARQL query without LIMIT/OFFSET clauses
    page_size : int — number of rows per request (default 5000)

    Returns
    -------
    pd.DataFrame with all results concatenated
    """
    results, offset = [], 0
    while True:
        paginated = query + f"\nLIMIT {page_size} OFFSET {offset}"
        try:
            df_page = run_query(PHAROS_ENDPOINT, paginated)
            if df_page.empty:
                break
            results.append(df_page)
            print(f"Fetched {offset + len(df_page)} rows...")
            if len(df_page) < page_size:
                break
            offset += page_size
            time.sleep(1)
        except Exception as e:
            print(f"Error at offset {offset}: {e}")
            time.sleep(3)
            continue
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

os.makedirs("data",   exist_ok=True)
print("Setup complete.")

Setup complete.


---
## 1. Data Collection — PHAROS

Persons are retrieved from artresearch.net using four UNION clauses covering the different
CIDOC-CRM modelling patterns used by the five active institutions:

| Institution | Modelling pattern |
|---|---|
| PMC, Frick, Zeri | `E21_Person / E39_Actor` + `P14i_performed` |
| MIDAS | `E39_Actor` + `E7_Activity` + `P14_carried_out_by` |
| Warburg | `E39_Actor` + `E12_Production` + `P14_carried_out_by` |

External identifiers are extracted from `sameAs` and `P141i_was_assigned_by` links:
Getty ULAN, Wikidata, VIAF, and GND (Deutsche Nationalbibliothek).

In [2]:
QUERY_ALL_PERSONS = """
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?person ?name ?ulanID ?wikidataID ?viafID ?institution
WHERE {
  {
    ?person a crm:E21_Person .
    ?person crm:P14i_performed ?activity .
    BIND(REPLACE(STR(?person), "https://artresearch.net/resource/([^/]+)/.*", "$1") AS ?institution)
  } UNION {
    ?person a crm:E39_Actor .
    ?person crm:P14i_performed ?activity .
    BIND(REPLACE(STR(?person), "https://artresearch.net/resource/([^/]+)/.*", "$1") AS ?institution)
  } UNION {
    ?person a crm:E39_Actor .
    ?activity crm:P14_carried_out_by ?person .
    ?activity a crm:E7_Activity .
    BIND(REPLACE(STR(?activity), "https://artresearch.net/resource/([^/]+)/.*", "$1") AS ?institution)
  } UNION {
    ?person a crm:E39_Actor .
    ?prod a crm:E12_Production .
    ?prod crm:P14_carried_out_by ?person .
    BIND(REPLACE(STR(?prod), "https://artresearch.net/resource/([^/]+)/.*", "$1") AS ?institution)
  }

  FILTER(?institution IN ("pmc", "midas", "frick", "zeri", "warburg"))

  OPTIONAL {
    ?person crm:P1_is_identified_by ?appellation .
    { ?appellation rdfs:label ?name }
    UNION
    { ?appellation <http://www.cidoc-crm.org/cidoc-crm/P190_has_symbolic_content> ?name }
  }

  OPTIONAL {
    { ?person <https://artresearch.net/custom/sameAs> ?ulanID . }
    UNION
    { ?person crm:P141i_was_assigned_by ?a . ?a crm:P141_assigned ?ulanID . }
    FILTER(CONTAINS(STR(?ulanID), "vocab.getty.edu/ulan"))
  }

  OPTIONAL {
    ?person <https://artresearch.net/custom/sameAs> ?wikidataID .
    FILTER(CONTAINS(STR(?wikidataID), "wikidata.org/entity"))
  }

  OPTIONAL {
    ?person <https://artresearch.net/custom/sameAs> ?viafID .
    FILTER(CONTAINS(STR(?viafID), "viaf.org"))
  }
}
"""

print("Fetching persons from PHAROS (paginated)...")
df_raw = fetch_all_paginated(QUERY_ALL_PERSONS)
df_raw.to_csv("data/persons_raw.csv", index=False)
print(f"Total rows: {len(df_raw)}")
print(df_raw["institution"].value_counts())

Fetching persons from PHAROS (paginated)...
Fetched 5000 rows...
Fetched 10000 rows...
Fetched 15000 rows...
Fetched 20000 rows...
Fetched 25000 rows...
Fetched 30000 rows...
Fetched 35000 rows...
Fetched 40000 rows...
Fetched 45000 rows...
Fetched 50000 rows...
Fetched 55000 rows...
Fetched 60000 rows...
Fetched 65000 rows...
Fetched 70000 rows...
Fetched 75000 rows...
Fetched 80000 rows...
Fetched 85000 rows...
Fetched 90000 rows...
Fetched 95000 rows...
Fetched 100000 rows...
Fetched 105000 rows...
Fetched 110000 rows...
Fetched 115000 rows...
Fetched 120000 rows...
Fetched 125000 rows...
Fetched 130000 rows...
Fetched 135000 rows...
Fetched 140000 rows...
Fetched 145000 rows...
Fetched 150000 rows...
Fetched 155000 rows...
Fetched 160000 rows...
Fetched 165000 rows...
Fetched 170000 rows...
Fetched 175000 rows...
Fetched 180000 rows...
Fetched 185000 rows...
Fetched 190000 rows...
Fetched 195000 rows...
Fetched 200000 rows...
Fetched 205000 rows...
Fetched 210000 rows...
Fetched 21

### 1.1 Cleaning & ID Extraction

In [3]:
df_raw = pd.read_csv("data/persons_raw.csv", low_memory=False)

# ── Step 1: Remove anonymous attributions in work production records ──────────
ANON_PATTERN = (
    "anonymous|copy of|shop of|school of|after |follower|circle of|"
    "manner of|anonimo|and assistants|attributed|workshop|studio of|forgery of"
)
before = len(df_raw)
df_raw = df_raw[
    ~df_raw["person"].str.contains("/work/", na=False) |
    (df_raw["person"].str.contains("/work/", na=False) &
     ~df_raw["name"].str.contains(ANON_PATTERN, case=False, na=True))
]
print(f"Removed anonymous work records: {before - len(df_raw)}")

# ── Step 2: Extract IDs from person URI ───────────────────────────────────────
def extract_id(uri):
    """
    Extract the ID type and value from a person URI.

    Recognises Getty ULAN, VIAF, Wikidata, GND, PHAROS artresearch.net,
    and Warburg sas.ac.uk URI patterns.

    Parameters
    ----------
    uri : str — person URI string

    Returns
    -------
    tuple (id_type: str, id_value: str)
    """
    uri = str(uri)
    if "vocab.getty.edu/ulan/" in uri:
        return ("ulan",     uri.split("vocab.getty.edu/ulan/")[-1])
    elif "viaf.org/viaf/" in uri:
        return ("viaf",     uri.split("viaf.org/viaf/")[-1])
    elif "wikidata.org/entity/" in uri:
        return ("wikidata", uri.split("wikidata.org/entity/")[-1])
    elif "d-nb.info/gnd/" in uri:
        return ("gnd",      uri.split("d-nb.info/gnd/")[-1])
    elif "artresearch.net/resource/" in uri:
        return ("pharos",   uri.split("artresearch.net/resource/")[-1])
    else:
        return ("unknown",  uri)

df_raw["id_type"], df_raw["id_value"] = zip(*df_raw["person"].apply(extract_id))

df_raw["ULAN_ID"] = df_raw.apply(
    lambda r: r["id_value"] if r["id_type"] == "ulan"
    else (str(r["ulanID"]).split("ulan/")[-1] if pd.notna(r["ulanID"]) else None), axis=1)
df_raw["VIAF_ID"] = df_raw.apply(
    lambda r: r["id_value"] if r["id_type"] == "viaf"
    else (str(r["viafID"]).split("viaf/")[-1] if pd.notna(r["viafID"]) else None), axis=1)
df_raw["WIKIDATA_ID"] = df_raw.apply(
    lambda r: r["id_value"] if r["id_type"] == "wikidata"
    else (str(r["wikidataID"]).split("entity/")[-1] if pd.notna(r["wikidataID"]) else None), axis=1)
df_raw["GND_ID"] = df_raw.apply(
    lambda r: r["id_value"] if r["id_type"] == "gnd" else None, axis=1)
df_raw = df_raw.drop(columns=["ulanID", "wikidataID", "viafID", "id_type", "id_value"])

# ── Step 3: Group name variants per person URI ────────────────────────────────
df_persons = df_raw.groupby("person").agg({
    "name":        lambda x: " | ".join(sorted(set(filter(lambda v: isinstance(v, str), x)))),
    "institution": lambda x: " | ".join(sorted(set(x))),
    "ULAN_ID":     "first",
    "VIAF_ID":     "first",
    "WIKIDATA_ID": "first",
    "GND_ID":      "first"
}).reset_index()
df_persons.rename(columns={"name": "allNames"}, inplace=True)
df_persons["allNames"] = df_persons["allNames"].replace("", "Unknown / Anonymous")

# ── Step 4: Deduplicate by external ID ───────────────────────────────────────
def get_group(row):
    if pd.notna(row["ULAN_ID"]):     return ("ULAN",     row["ULAN_ID"])
    if pd.notna(row["WIKIDATA_ID"]): return ("WIKIDATA", row["WIKIDATA_ID"])
    if pd.notna(row["VIAF_ID"]):     return ("VIAF",     row["VIAF_ID"])
    if pd.notna(row["GND_ID"]):      return ("GND",      row["GND_ID"])
    return ("NONE", row["person"])

df_persons["group_type"], df_persons["group_id"] = zip(*df_persons.apply(get_group, axis=1))

df_final = df_persons.groupby("group_id").agg({
    "person":      lambda x: " | ".join(sorted(set(x))),
    "allNames":    lambda x: " | ".join(sorted(set(" | ".join(x).split(" | ")))),
    "institution": lambda x: " | ".join(sorted(set(" | ".join(x).split(" | ")))),
    "ULAN_ID":     lambda x: " | ".join(sorted(set(filter(lambda v: isinstance(v, str), x)))),
    "VIAF_ID":     lambda x: " | ".join(sorted(set(filter(lambda v: isinstance(v, str), x)))),
    "WIKIDATA_ID": lambda x: " | ".join(sorted(set(filter(lambda v: isinstance(v, str), x)))),
    "GND_ID":      lambda x: " | ".join(sorted(set(filter(lambda v: isinstance(v, str), x)))),
    "group_type":  "first"
}).reset_index()

# Fix ULAN_ID float and remove broken ULAN rows
df_final["ULAN_ID"] = df_final["ULAN_ID"].apply(
    lambda x: str(int(float(x.split(" | ")[0]))) if pd.notna(x) and x != "" else None)
df_final = df_final[~((df_final["group_type"] == "ULAN") & (df_final["ULAN_ID"].isna()))]

# Clear person URI for rows that have an external ID
df_final["person"] = df_final.apply(
    lambda r: None if r["group_type"] != "NONE" else r["person"], axis=1)

df_final = df_final.drop(columns=["group_id"])

# ── Step 5: Remove non-person records ────────────────────────────────────────
for pattern in ["/work/", "/repository/", "/place/", "/provider/", "/pharos/actor/"]:
    df_final = df_final[~df_final["person"].str.contains(pattern, na=False)]

# ── Step 6: Group NONE by allNames+institution (same person, different URIs) ──
none_df  = df_final[df_final["group_type"] == "NONE"].copy()
other_df = df_final[df_final["group_type"] != "NONE"].copy()

none_named = none_df[~none_df["allNames"].str.contains("Unknown / Anonymous", na=False)]
none_anon  = none_df[none_df["allNames"].str.contains("Unknown / Anonymous", na=False)]

none_grouped = none_named.groupby(["allNames", "institution"], as_index=False).agg({
    "person":      lambda x: " | ".join(sorted(set(filter(lambda v: isinstance(v, str), x)))),
    "ULAN_ID":     "first",
    "VIAF_ID":     "first",
    "WIKIDATA_ID": "first",
    "GND_ID":      "first",
    "group_type":  "first"
})

df_final = pd.concat([other_df, none_grouped, none_anon], ignore_index=True)

# ── Step 7: Remove 'Unknown / Anonymous' from mixed allNames ─────────────────
df_final["allNames"] = df_final["allNames"].apply(
    lambda x: " | ".join([n for n in x.split(" | ") if n.strip() != "Unknown / Anonymous"])
    if isinstance(x, str) and "Unknown / Anonymous" in x and x != "Unknown / Anonymous" else x
)
df_final["allNames"] = df_final["allNames"].replace("", "Unknown / Anonymous")

# ── Step 8: Remove unidentified sources ──────────────────────────────────────
df_final = df_final[~df_final["allNames"].str.contains(
    "unidentified|unindentified|unknown source|unknown photographer",
    case=False, na=False
)]
df_final = df_final[~df_final["person"].str.contains(
    "purchased|gift_of|courtesy_of|from_negative|lent_by|made_at|spoiling",
    case=False, na=False
)]

# ── Step 9: Keep only persons with external ID for analysis ──────────────────
df_with_id = df_final[df_final["group_type"] != "NONE"].copy()

df_final.to_csv("data/all_persons_final.csv", index=False)
df_with_id.to_csv("data/persons_with_id.csv", index=False)

print(f"Total unique persons : {len(df_final)}")
print(f"With external ID     : {len(df_with_id)}")
print(df_final["group_type"].value_counts())
print(df_final["institution"].value_counts().head(10))

Removed anonymous work records: 31756
Total unique persons : 77956
With external ID     : 34346
group_type
NONE        43610
ULAN        24950
WIKIDATA     7699
VIAF         1558
GND           139
Name: count, dtype: int64
institution
midas                      36956
frick                      22463
pmc                         3828
warburg                     3809
zeri                        3589
frick | midas               1246
midas | zeri                1010
midas | warburg              939
frick | pmc                  874
frick | midas | warburg      686
Name: count, dtype: int64


---
## 2. Getty ULAN Enrichment

For persons with a ULAN ID, biographical data is retrieved from the Getty ULAN SPARQL endpoint:
nationality, gender, active period (estStart / estEnd), and birth / death places.

Getty data model path:
```
?ulanID → foaf:focus → ?agent
?agent  → gvp:nationalityPreferred → ?nat → skos:prefLabel → ?nationality
?agent  → gvp:biographyPreferred   → ?bio
?bio    → schema:gender            → AAT URI (300189559 = male, 300189557 = female)
?bio    → gvp:estStart / gvp:estEnd
?bio    → schema:birthPlace / schema:deathPlace → TGN URI → gvp:prefLabelGVP → ?place
```

In [4]:
df = pd.read_csv('data/persons_with_id.csv', low_memory=False, dtype={'ULAN_ID': str})
ulan_ids = df[df['ULAN_ID'].notna()]['ULAN_ID'].tolist()

batch_size = 100
batches = [ulan_ids[i:i+batch_size] for i in range(0, len(ulan_ids), batch_size)]
results = []

for i, batch in enumerate(batches):
    values = " ".join(f"<http://vocab.getty.edu/ulan/{uid}>" for uid in batch)
    query = f"""
PREFIX gvp:    <http://vocab.getty.edu/ontology#>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX foaf:   <http://xmlns.com/foaf/0.1/>
PREFIX schema: <http://schema.org/>

SELECT ?ulanID ?nationality ?gender ?estStart ?estEnd ?birthPlace ?deathPlace
WHERE {{
  VALUES ?ulanID {{ {values} }}
  ?ulanID foaf:focus ?agent .
  OPTIONAL {{
    ?agent gvp:nationalityPreferred ?nat .
    ?nat skos:prefLabel ?nationality .
    FILTER(LANG(?nationality) = "en")
  }}
  OPTIONAL {{
    ?agent gvp:biographyPreferred ?bio .
    OPTIONAL {{
      ?bio schema:gender ?genderURI .
      BIND(IF(?genderURI = <http://vocab.getty.edu/aat/300189559>, "male",
           IF(?genderURI = <http://vocab.getty.edu/aat/300189557>, "female", "unidentified")) AS ?gender)
    }}
    OPTIONAL {{ ?bio gvp:estStart ?estStart . }}
    OPTIONAL {{ ?bio gvp:estEnd   ?estEnd   . }}
    OPTIONAL {{
      ?bio schema:birthPlace ?bpPlace .
      BIND(IRI(REPLACE(STR(?bpPlace), "-place$", "")) AS ?bpURI)
      ?bpURI gvp:prefLabelGVP ?bpPref .
      ?bpPref <http://www.w3.org/2008/05/skos-xl#literalForm> ?birthPlace .
      FILTER(LANG(?birthPlace) = "en")
    }}
    OPTIONAL {{
      ?bio schema:deathPlace ?dpPlace .
      BIND(IRI(REPLACE(STR(?dpPlace), "-place$", "")) AS ?dpURI)
      ?dpURI gvp:prefLabelGVP ?dpPref .
      ?dpPref <http://www.w3.org/2008/05/skos-xl#literalForm> ?deathPlace .
      FILTER(LANG(?deathPlace) = "en")
    }}
  }}
}}
"""
    try:
        df_batch = run_query(GETTY_ENDPOINT, query)
        if not df_batch.empty:
            results.append(df_batch)
            print(f"Batch {i+1}: {len(df_batch)} rows")
        else:
            print(f"Batch {i+1}: empty")
    except Exception as e:
        print(f"Batch {i+1} failed: {e}")
    time.sleep(1)

    # Record every 50 batches.
    if (i+1) % 50 == 0:
        df_temp = pd.concat(results, ignore_index=True)
        df_temp.to_csv('data/getty_data.csv', index=False)
        print(f"Progress: {i+1}/{len(batches)}, saved {len(df_temp)} rows")

# Final record
df_getty = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
df_getty.to_csv('data/getty_data.csv', index=False)
print(f"Done: {len(df_getty)} results.")

Batch 1: 100 rows
Batch 2: 100 rows
Batch 3: 100 rows
Batch 4: 100 rows
Batch 5: 100 rows
Batch 6: 100 rows
Batch 7: 100 rows
Batch 8: 100 rows
Batch 9: 100 rows
Batch 10: 99 rows
Batch 11: 100 rows
Batch 12: 100 rows
Batch 13: 100 rows
Batch 14: 100 rows
Batch 15: 100 rows
Batch 16: 100 rows
Batch 17: 100 rows
Batch 18: 100 rows
Batch 19: 100 rows
Batch 20: 100 rows
Batch 21: 100 rows
Batch 22: 100 rows
Batch 23: 100 rows
Batch 24: 100 rows
Batch 25: 100 rows
Batch 26: 100 rows
Batch 27: 100 rows
Batch 28: 100 rows
Batch 29: 100 rows
Batch 30: 100 rows
Batch 31: 100 rows
Batch 32: 100 rows
Batch 33: 100 rows
Batch 34: 100 rows
Batch 35: 100 rows
Batch 36: 100 rows
Batch 37: 100 rows
Batch 38: 100 rows
Batch 39: 100 rows
Batch 40: 100 rows
Batch 41: 100 rows
Batch 42: 100 rows
Batch 43: 100 rows
Batch 44: 100 rows
Batch 45: 100 rows
Batch 46: 100 rows
Batch 47: 100 rows
Batch 48: 100 rows
Batch 49: 100 rows
Batch 50: 100 rows
Progress: 50/250, saved 4999 rows
Batch 51: 100 rows
Batch 5

In [5]:
df_final   = pd.read_csv('data/all_persons_final.csv', low_memory=False, dtype={'ULAN_ID': str})
df_getty   = pd.read_csv('data/getty_data.csv', low_memory=False)

df_getty['ulanID'] = df_getty['ulanID'].str.split('ulan/').str[-1]

# Merge
df_enriched = df_final.merge(df_getty, left_on='ULAN_ID', right_on='ulanID', how='left')
df_enriched = df_enriched.drop(columns=['ulanID'], errors='ignore')
df_enriched['estStart'] = pd.to_numeric(df_enriched['estStart'], errors='coerce').astype('Int64')
df_enriched['estEnd']   = pd.to_numeric(df_enriched['estEnd'],   errors='coerce').astype('Int64')

df_enriched.to_csv('data/persons_enriched.csv', index=False)
print(f"Total: {len(df_enriched)}")
print(f"With nationality : {df_enriched['nationality'].notna().sum()}")
print(f"With gender      : {df_enriched['gender'].notna().sum()}")
print(f"With estStart    : {df_enriched['estStart'].notna().sum()}")
print(f"With birthPlace  : {df_enriched['birthPlace'].notna().sum()}")
print(f"With deathPlace  : {df_enriched['deathPlace'].notna().sum()}")

Total: 77956
With nationality : 24817
With gender      : 24787
With estStart    : 24802
With birthPlace  : 5549
With deathPlace  : 5375


---
## RQ1 — Institution Distribution
> *How are persons distributed across PHAROS member institutions?*

In [ ]:
df = pd.read_csv("data/persons_enriched.csv", low_memory=False)
df_id = df[df["group_type"] != "NONE"].copy()  # only verified persons

INST_LABELS = {
    "midas":   "Bildarchiv Foto Marburg (MIDAS)",
    "frick":   "Frick Art Research Library",
    "zeri":    "Fondazione Zeri",
    "pmc":     "Paul Mellon Centre",
    "warburg": "Warburg Institute"
}

rows = []
for inst, label in INST_LABELS.items():
    mask = df_id["institution"].str.contains(inst, na=False)
    total = mask.sum()
    single = (df_id[mask]["institution"] == inst).sum()
    overlap = total - single
    rows.append({"institution": label, "single": single, "overlap": overlap, "total": total})

df_rq1 = pd.DataFrame(rows).sort_values("total", ascending=False)
df_rq1.to_csv("data/rq1_institutions.csv", index=False)

print(f"Verified persons (single institution): {(~df_id['institution'].str.contains('\\|', na=False)).sum()}")
print(f"Verified persons (multi-institution)  : {df_id['institution'].str.contains('\\|', na=False).sum()}")
print(df_rq1.to_string())

---
## RQ2 — Nationality Distribution
> *Which nationalities are represented in the PHAROS archives, and which dominate?*

Nationality sourced from Getty ULAN (`gvp:nationalityPreferred`).

In [8]:
df = pd.read_csv("data/persons_enriched.csv", low_memory=False)

df_nat = df["nationality"].value_counts().reset_index()
df_nat.columns = ["nationality", "count"]
df_nat = df_nat[df_nat["nationality"] != "undetermined (information indicator)"]
df_nat.to_csv("data/rq2_nationality.csv", index=False)

print(f"Unique nationalities: {len(df_nat)}")
print(df_nat.head(15).to_string())


Unique nationalities: 104
                        nationality  count
0        Italian (culture or style)   5655
1   German (culture, style, period)   4509
2                  British (modern)   4310
3         French (culture or style)   2708
4         American (North American)   1859
5          Dutch (culture or style)   1204
6        Flemish (culture or style)    623
7                          Austrian    609
8        Spanish (culture or style)    608
9                             Swiss    519
10                 Belgian (modern)    233
11                    Netherlandish    194
12       Russian (culture or style)    156
13        Danish (culture or style)    143
14                            Irish    138


---
## RQ3 — Gender Representation Over Time
> *What is the gender breakdown of persons in the archives, and how has it changed across decades?*

Gender sourced from Getty ULAN (AAT `300189559` = male, `300189557` = female).
Active period start year (`estStart`) is used as a proxy for decade grouping.

In [9]:
# RQ3 gender by decade
df_time = df.dropna(subset=['gender', 'estStart']).copy()

df_time['decade'] = (df_time['estStart'] // 10 * 10).astype(int)
df_time = df_time[(df_time['decade'] >= 1300) & (df_time['decade'] <= 1950)]
df_decade = df_time.groupby(['decade', 'gender']).size().reset_index(name='count')
df_decade.to_csv('data/rq3_gender_decade.csv', index=False)


In [ ]:
# RQ3 overall gender totals
# Note: raw Getty data uses "other" for unidentified gender records; relabelled for clarity
df_gender_total = df["gender"].replace({"other": "unidentified"}).value_counts().reset_index()
df_gender_total.columns = ["gender", "count"]
df_gender_total.to_csv("data/rq3_gender_total.csv", index=False)

print(df_gender_total.to_string())
print(f"\nMale share: {df_gender_total[df_gender_total['gender']=='male']['count'].values[0] / df_gender_total['count'].sum() * 100:.1f}%")

---
## RQ4 — Geographic Mobility
> *Where were persons born and where did they die — what does this reveal about geographic mobility?*

Birth and death places sourced from Getty ULAN TGN-linked place records.

In [10]:
# RQ4
df['birthPlace'].value_counts().head(20).reset_index().rename(columns={'birthPlace':'place','count':'count'}).to_csv('data/rq4_birthplace.csv', index=False)
df['deathPlace'].value_counts().head(20).reset_index().rename(columns={'deathPlace':'place','count':'count'}).to_csv('data/rq4_deathplace.csv', index=False)

df_both = df.dropna(subset=['birthPlace', 'deathPlace'])
moved  = (df_both['birthPlace'] != df_both['deathPlace']).sum()
stayed = (df_both['birthPlace'] == df_both['deathPlace']).sum()
print(f"\nMobility: {moved} moved ({moved/len(df_both)*100:.1f}%), {stayed} stayed ({stayed/len(df_both)*100:.1f}%)")


Mobility: 1716 moved (51.1%), 1639 stayed (48.9%)


In [ ]:
df = pd.read_csv('data/persons_enriched.csv', low_memory=False)

# only birthplace
only_birth = df[df['birthPlace'].notna() & df['deathPlace'].isna()]
# only deathplace
only_death = df[df['deathPlace'].notna() & df['birthPlace'].isna()]
# both
both = df[df['birthPlace'].notna() & df['deathPlace'].notna()]

print(f"Only birthPlace: {len(only_birth)}")
print(f"Only deathPlace: {len(only_death)}")
print(f"Both: {len(both)}")

# DeathPlace top cities — only deathPlace 
print("\nTop death cities (all with deathPlace):")
print(df['deathPlace'].value_counts().head(10))

print("\nTop birth cities (all with birthPlace):")
print(df['birthPlace'].value_counts().head(10))

Only birthPlace: 2194
Only deathPlace: 2020
Both: 3355

Top death cities (all with deathPlace):
deathPlace
London                   1065
Paris                    1024
New York                  272
Berlin (Berlin state)     237
Amsterdam                 195
Bologna                   118
Edinburgh                  97
Frankfurt am Main          72
Haarlem                    52
Dublin                     50
Name: count, dtype: int64

Top birth cities (all with birthPlace):
birthPlace
London                   712
Paris                    683
Amsterdam                160
Bologna                  154
Berlin (Berlin state)    146
New York                 115
Edinburgh                106
Philadelphia              81
Dublin                    67
Frankfurt am Main         66
Name: count, dtype: int64


---
## Conclusions

**RQ1 — Institution Distribution:**
Bildarchiv Foto Marburg (MIDAS) and the Frick Art Research Library together account for the
vast majority of verified persons. 7,311 persons appear in more than one institution,
indicating active cross-institutional linking via shared external identifiers (ULAN, Wikidata, VIAF, GND).

**RQ2 — Nationality:**
Italian persons dominate the archive (5,653), reflecting the Italian-focused holdings of
Fondazione Zeri and the Frick Art Research Library. German and British persons follow.
The distribution is heavily Western European, highlighting a geographic bias inherent to the current PHAROS corpus.

**RQ3 — Gender:**
Male persons account for 93.5% of verified persons. Female representation remains below 5%
across most periods, with a modest increase from the 1840s onward. This reflects both
historical barriers to women in the arts and possible archival documentation gaps.

**RQ4 — Geographic Mobility:**
London and Paris dominate as death cities far more than as birth cities, confirming their role
as gravitational centres of the art world. Of persons with both places recorded, the majority
died in a different city from where they were born.

---

## Credits & Licenses

| Source | License |
|---|---|
| artresearch.net (PHAROS) | [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) |
| Getty ULAN | [Open Data Commons Attribution License](http://opendatacommons.org/licenses/by/1.0/) |

**Author:** Yiğit Ak  
**Course:** Information Visualization course (a.a. 2025/2026) within the Digital Humanities and Digital Knowledge Master's Degree at Alma Mater Studiorum - University of Bologna.